# 🌱 GrowMind — Entrenamiento del modelo propio de salud de plantas

Notebook listo para correr en **Google Colab** (GPU gratis: Runtime → Change runtime type → T4 GPU).

**Objetivo:** entrenar un MobileNetV3 que clasifique fotos de hojas de plantas en categorías de salud / deficiencia, y exportarlo a TensorFlow Lite para correr 100% on-device en el celu (sin internet, sin API key, gratis para siempre).

**Estrategia:**
1. Bajar el dataset **PlantVillage** (38 clases, 54k imágenes, libre).
2. Hacer transfer learning desde MobileNetV3-Small pre-entrenado en ImageNet.
3. Fine-tunear con cannabis-specific data via **Roboflow Universe** (free).
4. Exportar a TFLite (~4MB) para mobile.
5. Subir el .tflite al repo y cargarlo en la PWA con TensorFlow.js.

**Costo total: USD 0.** Colab es gratis, los datasets son gratis, y el modelo final corre localmente sin costo de inferencia.

## 1. Setup

In [ ]:
!pip install -q tensorflow tensorflow-datasets pillow matplotlib roboflow

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import os

print('TF version:', tf.__version__)
print('GPU disponible:', len(tf.config.list_physical_devices('GPU')) > 0)

## 2. Bajar PlantVillage (dataset libre, 54k fotos)

PlantVillage tiene 38 clases que cubren las enfermedades visuales más comunes en cultivos. Lo usamos como base general — el modelo va a aprender qué es "hoja sana" vs "hoja con problema" en general antes de fine-tunear con cannabis específicamente.

In [ ]:
# Bajar PlantVillage de TensorFlow Datasets (oficial, libre)
(ds_train, ds_val), ds_info = tfds.load(
    'plant_village',
    split=['train[:80%]', 'train[80%:]'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

NUM_CLASSES = ds_info.features['label'].num_classes
CLASS_NAMES = ds_info.features['label'].names
print(f'Clases: {NUM_CLASSES}')
print('Primeras 10:', CLASS_NAMES[:10])

## 3. Preprocesamiento + augmentation

In [ ]:
IMG_SIZE = 224
BATCH = 64

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
    tf.keras.layers.RandomBrightness(0.15),
])

train = (ds_train
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(lambda x, y: (augment(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(2000)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE))

val = (ds_val
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE))

## 4. Modelo: MobileNetV3-Small con transfer learning

MobileNetV3-Small es ideal para mobile: ~2.5M parámetros, ~4MB en TFLite, corre en <100ms en cualquier celular moderno.

In [ ]:
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg',
)
base.trainable = False  # primera fase: solo el head

model = tf.keras.Sequential([
    base,
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 5. Entrenamiento (fase 1: head only)

In [ ]:
history1 = model.fit(
    train,
    validation_data=val,
    epochs=8,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    ],
)

## 6. Fine-tuning (fase 2: descongelar las últimas capas)

In [ ]:
base.trainable = True
# Congelar las primeras capas, fine-tunear solo las últimas ~30
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # LR mucho más bajo
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history2 = model.fit(
    train,
    validation_data=val,
    epochs=6,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    ],
)

## 7. (Opcional) Fine-tuning específico de cannabis con Roboflow Universe

Roboflow Universe tiene varios datasets públicos de cannabis (deficiencias N/P/K, plagas, etapas de floración). Buscá "cannabis deficiency" en https://universe.roboflow.com y conseguí tu API key gratis en https://app.roboflow.com.

Reemplazá los placeholders y ejecutá esta celda solo si querés especialización en cannabis (recomendado pero opcional).

In [ ]:
# from roboflow import Roboflow
# rf = Roboflow(api_key='TU_API_KEY')
# project = rf.workspace('WORKSPACE').project('PROJECT_SLUG')
# dataset = project.version(1).download('folder')
# # Después: tf.keras.utils.image_dataset_from_directory(dataset.location, ...)
# # y un par de epochs más con LR bajo

## 8. Exportar a TensorFlow Lite (mobile-ready)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantization para que sea liviano
tflite_model = converter.convert()

with open('plant_health.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize('plant_health.tflite') / 1024 / 1024
print(f'✓ Modelo exportado: plant_health.tflite ({size_mb:.2f} MB)')

# Guardar también las labels
with open('labels.json', 'w') as f:
    import json
    json.dump(CLASS_NAMES, f, indent=2)
print('✓ Labels guardadas en labels.json')

## 9. Bajar archivos

Descargá `plant_health.tflite` y `labels.json` y subilos a `growmind-mobile/public/models/` en tu repo. Después en la app, podés cargar el modelo con TensorFlow.js (`@tensorflow/tfjs` + `@tensorflow/tfjs-tflite`).

In [ ]:
from google.colab import files
files.download('plant_health.tflite')
files.download('labels.json')

## Próximos pasos en la app

1. Crear `growmind-mobile/public/models/plant_health.tflite`.
2. Instalar `@tensorflow/tfjs @tensorflow/tfjs-tflite` en el proyecto.
3. Crear un helper `analyzeLocalTflite(imageDataUrl)` que cargue el modelo una sola vez (cache) y devuelva la clase + confianza.
4. En `analyzeWithGemini()`, intentar primero el TFLite local; si la confianza es alta, usarlo (gratis, offline). Si es baja, fallback a Gemini para razonamiento más profundo.

Esto te da el **híbrido completo**: modelo propio rápido y gratis para 80% de los casos, Gemini para el 20% más complejo o ambiguo.